# STLRocket — results visualization

Compares experiment results across datasets, `n_formulas`, and `depth_max`.
Each subdirectory of `results/` is one run; parameters are read from `config.json`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="tab10")
RESULTS_DIR = Path("results")

## 1 · Load results

In [ ]:
def load_all_results(results_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Scan results_dir and return three tidy DataFrames:
      - acc_df   : one row per (run, dataset, n_formulas, depth_max)
      - local_df : local_explanations.csv rows with config columns added
      - global_df: global_explanations.csv rows with config columns added
    """
    acc_rows, local_rows, global_rows = [], [], []

    for run_dir in sorted(results_dir.iterdir()):
        if not run_dir.is_dir():
            continue
        config_path = run_dir / "config.json"
        acc_path    = run_dir / "accuracy.json"
        if not config_path.exists() or not acc_path.exists():
            continue

        cfg = json.loads(config_path.read_text())
        acc = json.loads(acc_path.read_text())

        key = dict(
            dataset    = cfg["dataset"],
            n_formulas = cfg["n_formulas"],
            depth_max  = cfg["depth_max"],
            run_dir    = run_dir.name,
        )

        # --- per-run accuracy rows ---
        for r in acc["per_run"]:
            row = {**key, "run": r["run"], "seed": r["seed"],
                   "balanced_accuracy": r["balanced_accuracy"],
                   "macro_f1": r["macro_f1"],
                   "time_features_s": r["time_features_s"],
                   "time_train_s": r["time_train_s"],
                   "time_total_s": r["time_total_s"]}
            # flatten per-class metrics
            for cls, m in r.get("per_class", {}).items():
                for metric, val in m.items():
                    row[f"cls_{cls}_{metric}"] = val
            acc_rows.append(row)

        # --- local explanations ---
        local_path = run_dir / "local_explanations.csv"
        if local_path.exists():
            df = pd.read_csv(local_path)
            for k, v in key.items():
                df[k] = v
            local_rows.append(df)

        # --- global explanations ---
        global_path = run_dir / "global_explanations.csv"
        if global_path.exists():
            df = pd.read_csv(global_path)
            for k, v in key.items():
                df[k] = v
            global_rows.append(df)

    acc_df    = pd.DataFrame(acc_rows)    if acc_rows    else pd.DataFrame()
    local_df  = pd.concat(local_rows)    if local_rows  else pd.DataFrame()
    global_df = pd.concat(global_rows)   if global_rows else pd.DataFrame()
    return acc_df, local_df, global_df


acc_df, local_df, global_df = load_all_results(RESULTS_DIR)

if acc_df.empty:
    print("No results found. Run run_experiment.py first.")
else:
    print(f"Loaded {len(acc_df)} run-level rows across "
          f"{acc_df['dataset'].nunique()} dataset(s), "
          f"{acc_df['n_formulas'].nunique()} n_formulas value(s), "
          f"{acc_df['depth_max'].nunique()} depth_max value(s).")
    display(acc_df.groupby(["dataset","n_formulas","depth_max"])["balanced_accuracy"].agg(["mean","std"]).round(4))

## 2 · Balanced accuracy — n_formulas × depth_max heatmap

One heatmap per dataset. Rows = n_formulas values, columns = depth_max values.

In [ ]:
if not acc_df.empty:
    summary = (
        acc_df
        .groupby(["dataset", "n_formulas", "depth_max"])["balanced_accuracy"]
        .mean()
        .reset_index()
    )

    datasets = sorted(summary["dataset"].unique())
    ncols = min(3, len(datasets))
    nrows = int(np.ceil(len(datasets) / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)
    axes_flat = axes.flatten()

    for ax, ds in zip(axes_flat, datasets):
        sub = summary[summary["dataset"] == ds]
        pivot = sub.pivot(index="n_formulas", columns="depth_max", values="balanced_accuracy")
        pivot = pivot.sort_index(ascending=False)
        sns.heatmap(
            pivot, ax=ax, vmin=0, vmax=1, cmap="YlGn",
            annot=True, fmt=".3f", linewidths=0.4,
            cbar_kws={"label": "Balanced accuracy"},
        )
        ax.set_title(ds, fontsize=10, fontweight="bold")
        ax.set_xlabel("depth_max")
        ax.set_ylabel("n_formulas")

    for ax in axes_flat[len(datasets):]:
        ax.set_visible(False)

    fig.suptitle("Balanced accuracy — n_formulas × depth_max", fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

## 3 · Training time vs depth_max

Stacked bars: feature generation time (bottom) + linear model time (top).

In [ ]:
if not acc_df.empty:
    time_summary = (
        acc_df
        .groupby(["dataset", "n_formulas", "depth_max"])[["time_features_s", "time_train_s"]]
        .mean()
        .reset_index()
    )

    datasets = sorted(time_summary["dataset"].unique())
    n_formulas_vals = sorted(time_summary["n_formulas"].unique())
    depth_vals = sorted(time_summary["depth_max"].unique())

    ncols = min(3, len(datasets))
    nrows = int(np.ceil(len(datasets) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows),
                              squeeze=False)
    axes_flat = axes.flatten()

    bar_width = 0.8 / len(n_formulas_vals)

    for ax, ds in zip(axes_flat, datasets):
        sub = time_summary[time_summary["dataset"] == ds]
        x = np.arange(len(depth_vals))
        for i, nf in enumerate(n_formulas_vals):
            s = sub[sub["n_formulas"] == nf].sort_values("depth_max")
            offset = (i - len(n_formulas_vals) / 2 + 0.5) * bar_width
            ax.bar(x + offset, s["time_features_s"].values, bar_width,
                   label=f"n={nf} features", alpha=0.85)
            ax.bar(x + offset, s["time_train_s"].values, bar_width,
                   bottom=s["time_features_s"].values,
                   label=f"n={nf} train", alpha=0.5, hatch="//")
        ax.set_title(ds, fontsize=10)
        ax.set_xlabel("depth_max")
        ax.set_ylabel("seconds" if ax == axes_flat[0] else "")
        ax.set_xticks(x)
        ax.set_xticklabels(depth_vals)

    for ax in axes_flat[len(datasets):]:
        ax.set_visible(False)

    # Build a compact legend: solid = features, hatched = train
    from matplotlib.patches import Patch
    palette = sns.color_palette("tab10", len(n_formulas_vals))
    legend_handles = []
    for color, nf in zip(palette, n_formulas_vals):
        legend_handles.append(Patch(facecolor=color, label=f"n={nf} features"))
        legend_handles.append(Patch(facecolor=color, hatch="//", alpha=0.5, label=f"n={nf} train"))
    fig.legend(handles=legend_handles, loc="lower right", ncol=2, fontsize=8)
    fig.suptitle("Training time vs depth_max (solid=features, hatched=linear model)",
                 fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

## 4 · Per-class F1 heatmap

One heatmap per (n_formulas, depth_max) combination. Rows = datasets, columns = classes.

In [ ]:
if not acc_df.empty:
    # Extract per-class F1 columns
    cls_f1_cols = [c for c in acc_df.columns if c.startswith("cls_") and c.endswith("_f1")]
    if not cls_f1_cols:
        print("No per-class F1 data found.")
    else:
        cls_names = [c[4:-3] for c in cls_f1_cols]  # strip "cls_" prefix and "_f1" suffix

        per_cls_mean = (
            acc_df
            .groupby(["dataset", "n_formulas", "depth_max"])[cls_f1_cols]
            .mean()
            .reset_index()
        )

        combos = per_cls_mean[["n_formulas", "depth_max"]].drop_duplicates().sort_values(["n_formulas", "depth_max"])

        for _, combo in combos.iterrows():
            sub = per_cls_mean[
                (per_cls_mean["n_formulas"] == combo["n_formulas"]) &
                (per_cls_mean["depth_max"] == combo["depth_max"])
            ].set_index("dataset")[cls_f1_cols]
            sub.columns = cls_names
            sub = sub.dropna(axis=1, how="all")

            fig, ax = plt.subplots(figsize=(max(6, len(sub.columns) * 0.9), max(3, len(sub) * 0.5)))
            sns.heatmap(sub.astype(float), ax=ax, vmin=0, vmax=1, cmap="YlGn",
                        annot=True, fmt=".2f", linewidths=0.4,
                        cbar_kws={"label": "F1"})
            ax.set_title(f"Per-class F1 — n_formulas={int(combo['n_formulas'])}  depth_max={int(combo['depth_max'])}",
                         fontweight="bold")
            ax.set_xlabel("Class")
            ax.set_ylabel("Dataset")
            plt.tight_layout()
            plt.show()

## 5 · Local explanation quality — mean precision and mean support

Mean `precision_train` and `support` per dataset, across n_formulas and depth_max configurations.

In [ ]:
if local_df.empty:
    print("No local explanation data found.")
else:
    available = [c for c in ("precision_train", "support") if c in local_df.columns]
    if not available:
        print(f"Columns not found. Available: {list(local_df.columns)}")
    else:
        summary = (
            local_df
            .groupby(["dataset", "n_formulas", "depth_max"])[available]
            .mean()
            .reset_index()
        )

        datasets = sorted(summary["dataset"].unique())
        n_formulas_vals = sorted(summary["n_formulas"].unique())
        palette = sns.color_palette("tab10", len(n_formulas_vals))

        labels = {"precision_train": "Mean precision (train)", "support": "Mean support"}

        ncols = min(3, len(datasets))
        nrows = int(np.ceil(len(datasets) / ncols))

        for metric in available:
            fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows),
                                      squeeze=False, sharey=False)
            axes_flat = axes.flatten()

            for ax, ds in zip(axes_flat, datasets):
                sub = summary[summary["dataset"] == ds]
                for color, nf in zip(palette, n_formulas_vals):
                    s = sub[sub["n_formulas"] == nf].sort_values("depth_max")
                    ax.plot(s["depth_max"], s[metric], marker="o", color=color, label=f"n={nf}")
                ax.set_title(ds, fontsize=10, fontweight="bold")
                ax.set_xlabel("depth_max")
                ax.set_ylabel(labels[metric] if ax == axes_flat[0] else "")
                ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

            for ax in axes_flat[len(datasets):]:
                ax.set_visible(False)

            handles, labels_ = axes_flat[0].get_legend_handles_labels()
            fig.legend(handles, labels_, title="n_formulas",
                       loc="lower right", ncol=len(n_formulas_vals))
            fig.suptitle(f"{labels[metric]} vs depth_max", fontweight="bold", y=1.01)
            plt.tight_layout()
            plt.show()

## 6 · Global explanation F1 per class

Per-class F1 for each dataset, grouped by (n_formulas, depth_max) configuration.

In [ ]:
if global_df.empty:
    print("No global explanation data found.")
else:
    datasets = sorted(global_df["dataset"].unique())

    for ds in datasets:
        sub = global_df[global_df["dataset"] == ds].copy()
        classes = sorted(sub["class"].unique())

        sub["config"] = sub.apply(
            lambda r: f"n={int(r['n_formulas'])} d={int(r['depth_max'])}", axis=1
        )
        configs = sorted(sub["config"].unique())
        palette = sns.color_palette("tab10", len(configs))

        x = np.arange(len(classes))
        bar_width = 0.8 / max(len(configs), 1)

        fig, ax = plt.subplots(figsize=(max(6, len(classes) * len(configs) * 0.7 + 2), 4))

        for i, (cfg, color) in enumerate(zip(configs, palette)):
            s = sub[sub["config"] == cfg].set_index("class")
            vals = [s.loc[c, "f1"] if c in s.index else np.nan for c in classes]
            offset = (i - len(configs) / 2 + 0.5) * bar_width
            ax.bar(x + offset, vals, bar_width, label=cfg, color=color, alpha=0.85)

        ax.set_xticks(x)
        ax.set_xticklabels(classes, rotation=30, ha="right")
        ax.set_ylim(0, 1)
        ax.set_ylabel("F1")
        ax.set_xlabel("Class")
        ax.legend(title="Configuration", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
        ax.set_title(f"Global explanation F1 per class — {ds}", fontweight="bold")
        plt.tight_layout()
        plt.show()